In [123]:
import tensorflow as tf
import numpy as np
import math
import os

In [124]:
# from tf.keras.models import Model  # This does not work!
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, GRU, Embedding
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, TensorBoard
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from europarl import maybe_download_and_extract
import europarl

In [4]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))


[]


In [5]:
tf.__version__

'2.16.1'

### loading data and saving it to numpy tensors

In [126]:
language_code='de'

In [127]:
mark_start = 'ssss '
mark_end = ' eeee'

In [128]:
maybe_download_and_extract(language_code=language_code)

Data has apparently already been downloaded and unpacked.


In [129]:

data_src = europarl.load_data(english=True,
                               language_code=language_code,
                               )

data_dest = europarl.load_data(english=False,
                              language_code=language_code,
                              start=mark_start,
                              end=mark_end)

In [130]:
len(data_src),len(data_dest)

(1920209, 1920209)

In [131]:
print(data_src[:3])
print(data_dest[:3])

['Resumption of the session', 'I declare resumed the session of the European Parliament adjourned on Friday 17 December 1999, and I would like once again to wish you a happy new year in the hope that you enjoyed a pleasant festive period.', "Although, as you will have seen, the dreaded 'millennium bug' failed to materialise, still the people in a number of countries suffered a series of natural disasters that truly were dreadful."]
['ssss Wiederaufnahme der Sitzungsperiode eeee', 'ssss Ich erkläre die am Freitag, dem 17. Dezember unterbrochene Sitzungsperiode des Europäischen Parlaments für wiederaufgenommen, wünsche Ihnen nochmals alles Gute zum Jahreswechsel und hoffe, daß Sie schöne Ferien hatten. eeee', 'ssss Wie Sie feststellen konnten, ist der gefürchtete "Millenium-Bug " nicht eingetreten. Doch sind Bürger einiger unserer Mitgliedstaaten Opfer von schrecklichen Naturkatastrophen geworden. eeee']


In [132]:
num_words = 10000

In [133]:
data_src=data_src[:50000]
data_dest=data_dest[:50000]

In [134]:
class TokenizerWrap(Tokenizer):
    """Wrap the Tokenizer-class from Keras with more functionality."""
    
    def __init__(self, texts, padding,
                 reverse=False, num_words=None):
        """
        :param texts: List of strings. This is the data-set.
        :param padding: Either 'post' or 'pre' padding.
        :param reverse: Boolean whether to reverse token-lists.
        :param num_words: Max number of words to use.
        """

        Tokenizer.__init__(self, num_words=num_words)

        # Create the vocabulary from the texts.
        self.fit_on_texts(texts)

        # Create inverse lookup from integer-tokens to words.
        self.index_to_word = dict(zip(self.word_index.values(),
                                      self.word_index.keys()))

        # Convert all texts to lists of integer-tokens.
        # Note that the sequences may have different lengths.
        self.tokens = self.texts_to_sequences(texts)

        if reverse:
            # Reverse the token-sequences.
            self.tokens = [list(reversed(x)) for x in self.tokens]
        
            # Sequences that are too long should now be truncated
            # at the beginning, which corresponds to the end of
            # the original sequences.
            truncating = 'pre'
        else:
            # Sequences that are too long should be truncated
            # at the end.
            truncating = 'post'

        # The number of integer-tokens in each sequence.
        self.num_tokens = [len(x) for x in self.tokens]

        # Max number of tokens to use in all sequences.
        # We will pad / truncate all sequences to this length.
        # This is a compromise so we save a lot of memory and
        # only have to truncate maybe 5% of all the sequences.
        self.max_tokens = np.mean(self.num_tokens) \
                          + 2 * np.std(self.num_tokens)
        self.max_tokens = int(self.max_tokens)

        # Pad / truncate all token-sequences to the given length.
        # This creates a 2-dim numpy matrix that is easier to use.
        self.tokens_padded = pad_sequences(self.tokens,
                                           maxlen=self.max_tokens,
                                           padding=padding,
                                           truncating=truncating)

    def token_to_word(self, token):
        """Lookup a single word from an integer-token."""

        word = " " if token == 0 else self.index_to_word[token]
        return word 

    def tokens_to_string(self, tokens):
        """Convert a list of integer-tokens to a string."""

        # Create a list of the individual words.
        words = [self.index_to_word[token]
                 for token in tokens
                 if token != 0]
        
        # Concatenate the words to a single string
        # with space between all the words.
        text = " ".join(words)

        return text
    
    def text_to_tokens(self, text, reverse=False, padding=False):
        """
        Convert a single text-string to tokens with optional
        reversal and padding.
        """

        # Convert to tokens. Note that we assume there is only
        # a single text-string so we wrap it in a list.
        tokens = self.texts_to_sequences([text])
        tokens = np.array(tokens)

        if reverse:
            # Reverse the tokens.
            tokens = np.flip(tokens, axis=1)

            # Sequences that are too long should now be truncated
            # at the beginning, which corresponds to the end of
            # the original sequences.
            truncating = 'pre'
        else:
            # Sequences that are too long should be truncated
            # at the end.
            truncating = 'post'

        if padding:
            # Pad and truncate sequences to the given length.
            tokens = pad_sequences(tokens,
                                   maxlen=self.max_tokens,
                                   padding='pre',
                                   truncating=truncating)

        return tokens

In [135]:
%%time
tokenizer_src = TokenizerWrap(texts=data_src,
                              padding='pre',
                              reverse=True,
                              num_words=num_words)

CPU times: total: 359 ms
Wall time: 4.26 s


In [136]:
%%time
tokenizer_dest = TokenizerWrap(texts=data_dest,
                               padding='post',
                               reverse=False,
                               num_words=num_words)

CPU times: total: 516 ms
Wall time: 5.78 s


In [137]:
tokens_src = tokenizer_src.tokens_padded
tokens_dest = tokenizer_dest.tokens_padded
print(tokens_src.shape)
print(tokens_dest.shape)

(50000, 55)
(50000, 50)


In [138]:
#This is the integer-token used to mark the beginning of a text in the destination-language.
token_start = tokenizer_dest.word_index[mark_start.strip()]
print(token_start)

2


In [139]:
#This is the integer-token used to mark the end of a text in the destination-language.
token_end = tokenizer_dest.word_index[mark_end.strip()]
token_end

3

In [140]:
tokens_src[1]

array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,  464, 6889,    7, 4140,
         40,    8,  196,    1,    5,  161,   75, 1320,    7,   40,  347,
          3,  235,  315,   52,   27,   12,    4,  218, 1167, 1954, 1600,
         13, 3827,   45,   24,    1,    2,  931,    1, 2523, 2522,   12],
      dtype=int32)

In [141]:
tokenizer_src.tokens_to_string(tokens_src[1])

'period pleasant a enjoyed you that hope the in year new happy a you wish to again once like would i and 1999 december 17 friday on adjourned parliament european the of session the resumed declare i'

### Preparing training data for Neutral Network

In [142]:
encoder_input_data = tokens_src

In [143]:
#The input and output data for the decoder is identical, 
# except shifted one time-step. We can use the same numpy array to save memory by slicing it
decoder_input_data = tokens_dest[:, :-1]
print(decoder_input_data.shape)

decoder_output_data = tokens_dest[:, 1:]
print(decoder_output_data.shape)

(50000, 49)
(50000, 49)


In [144]:
decoder_output_data

array([[3375,    4, 2204, ...,    0,    0,    0],
       [  12, 4135,    1, ...,    0,    0,    0],
       [  33,   25,  794, ...,    0,    0,    0],
       ...,
       [ 678,    7,  832, ...,    0,    0,    0],
       [  40,   65,   12, ...,    0,    0,    0],
       [ 269,  214,   12, ...,    0,    0,    0]], dtype=int32)

In [145]:
tokenizer_dest.tokens_to_string(decoder_input_data[1])


'ssss ich erkläre die am freitag dem 17 dezember unterbrochene sitzungsperiode des europäischen parlaments für wiederaufgenommen wünsche ihnen nochmals alles gute zum und hoffe daß sie schöne hatten eeee'

In [146]:
tokenizer_dest.tokens_to_string(decoder_output_data[1])


'ich erkläre die am freitag dem 17 dezember unterbrochene sitzungsperiode des europäischen parlaments für wiederaufgenommen wünsche ihnen nochmals alles gute zum und hoffe daß sie schöne hatten eeee'

### Create Neural Network

#### Create Encoder

In [147]:
encoder_input = Input(shape=(None, ), name='encoder_input')

In [148]:
embedding_size = 128

In [149]:
encoder_embedding = Embedding(input_dim=num_words,
                              output_dim=embedding_size,
                              name='encoder_embedding')

In [150]:
state_size = 512

In [151]:
encoder_gru1 = GRU(state_size, name='encoder_gru1',
                   return_sequences=True)
encoder_gru2 = GRU(state_size, name='encoder_gru2',
                   return_sequences=True)
encoder_gru3 = GRU(state_size, name='encoder_gru3',
                   return_sequences=False)

In [152]:
def connect_encoder():
    # Start the neural network with its input-layer.
    net = encoder_input
    
    # Connect the embedding-layer.
    net = encoder_embedding(net)

    # Connect all the GRU-layers.
    net = encoder_gru1(net)
    net = encoder_gru2(net)
    net = encoder_gru3(net)

    # This is the output of the encoder.
    encoder_output = net
    
    return encoder_output

In [153]:
encoder_output = connect_encoder()   #this is our context vectore generated by encoder

#### Create Decoder

The decoder takes two inputs. First it needs the "thought vector" produced by the encoder which summarizes the contents of the input-text.
The decoder also needs a sequence of integer-tokens as inputs. During training we will supply this with a full sequence of integer-tokens e.g. corresponding to the text "ssss once upon a time eeee".
During inference when we are translating new input-texts, we will start by feeding a sequence with just one integer-token for "ssss" which marks the beginning of a text, and combined with the "thought vector" from the encoder, the decoder will hopefully be able to produce the correct next word e.g. "once"

In [154]:
decoder_initial_state = Input(shape=(state_size,),name='decoder_initial_state')

In [155]:
decoder_input = Input(shape=(None, ), name='decoder_input')

In [156]:
decoder_embedding = Embedding(input_dim=num_words,
                              output_dim=embedding_size,
                              name='decoder_embedding')

In [157]:
decoder_gru1 = GRU(state_size, name='decoder_gru1',
                   return_sequences=True)
decoder_gru2 = GRU(state_size, name='decoder_gru2',
                   return_sequences=True)
decoder_gru3 = GRU(state_size, name='decoder_gru3',
                   return_sequences=True)

In [158]:
decoder_dense = Dense(num_words,
                      activation='softmax',
                      name='decoder_output')

In [159]:
def connect_decoder(initial_state):
    # Start the decoder-network with its input-layer.
    net = decoder_input

    # Connect the embedding-layer.
    net = decoder_embedding(net)
    
    # Connect all the GRU-layers.
    net = decoder_gru1(net, initial_state=initial_state)
    net = decoder_gru2(net, initial_state=initial_state)
    net = decoder_gru3(net, initial_state=initial_state)

    # Connect the final dense layer that converts to
    # one-hot encoded arrays.
    decoder_output = decoder_dense(net)
    
    return decoder_output

#### Create encoder-decoder Model

In [175]:
decoder_output = connect_decoder(initial_state=encoder_output)

model_train = Model(inputs=[encoder_input, decoder_input],
                    outputs=[decoder_output])

### Model Training

In [176]:
decoder_output = connect_decoder(initial_state=decoder_initial_state)

model_decoder = Model(inputs=[decoder_input, decoder_initial_state],
                      outputs=[decoder_output])

In [177]:
model_train.compile(optimizer='adam', 
                    loss='sparse_categorical_crossentropy', 
                    metrics=['accuracy'])


In [178]:
path_checkpoint = 'encoder_decoder_ashish.weights.h5'
callback_checkpoint = ModelCheckpoint(filepath=path_checkpoint,
                                      monitor='val_loss',
                                      verbose=1,
                                      save_weights_only=True,
                                      save_best_only=True)

In [179]:
callback_early_stopping = EarlyStopping(monitor='val_loss',
                                        patience=3, verbose=1)

In [180]:
callback_tensorboard = TensorBoard(log_dir='./ashish_logs/',
                                   histogram_freq=0,
                                   write_graph=False)

In [181]:
callbacks = [callback_early_stopping,
             callback_checkpoint,
             callback_tensorboard]

In [182]:
#We wrap the data in named dicts so we are sure the data is assigned correctly to the inputs and outputs of the model.
x_data = \
{
    'encoder_input': encoder_input_data,
    'decoder_input': decoder_input_data
}

y_data = \
{
    'decoder_output': decoder_output_data
}

In [183]:
decoder_output_data

array([[3375,    4, 2204, ...,    0,    0,    0],
       [  12, 4135,    1, ...,    0,    0,    0],
       [  33,   25,  794, ...,    0,    0,    0],
       ...,
       [ 678,    7,  832, ...,    0,    0,    0],
       [  40,   65,   12, ...,    0,    0,    0],
       [ 269,  214,   12, ...,    0,    0,    0]], dtype=int32)

In [184]:
print(type(decoder_output_data))  # Should be numpy.ndarray
print(decoder_output_data.dtype)  # Should be int32
print(decoder_output_data[:5])  # Print first 5 rows


<class 'numpy.ndarray'>
int32
[[3375    4 2204    3    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0]
 [  12 4135    1  109 1449   26 1390 1074 7254 2204   18   38  122   14
  2687 1615  108  803  351  478   59    5  308   10   25 6433  519    3
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0]
 [  33   25  794 1098   15    4   16 4694  102   31  150  860  121   79
   936   13 4009 3209  969    3    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0]
 [  21   64  209    4 1361   52   43  239   21 2537   44 2204    6    8
   428 1391    3    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0   

In [185]:
validation_split = 1000 / len(encoder_input_data)
validation_split

0.02

In [ ]:
model_train.fit(
    [encoder_input_data, decoder_input_data],  # Input data for both encoder and decoder
    decoder_output_data,  # Output data for the decoder
    batch_size=384,
    epochs=2,
    validation_split=0.1,  # For validation split, set as a percentage
    callbacks=callbacks
)


Epoch 1/2
  1/118 ━━━━━━━━━━━━━━━━━━━━ 4:15:43 131s/step - accuracy: 0.0000e+00 - loss: 9.2112

In [2]:
import torch

print("Number of GPU: ", torch.cuda.device_count())
print("GPU Name: ", torch.cuda.get_device_name())


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Number of GPU:  1
GPU Name:  NVIDIA GeForce RTX 3050 6GB Laptop GPU
Using device: cuda
